<a href="https://colab.research.google.com/github/shoh0806/Capstone_Design/blob/main/SASRec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  1. import

In [14]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import optim

# 2. 하이퍼 파라미터

In [15]:
max_len = 50
hidden_dim = 50
num_heads = 2
num_layers = 4
batch_size = 128
LR = 0.001
epochs = 30


# 3. 데이터 전처리

In [16]:
ratings = pd.read_csv("ratings.csv")
ratings = ratings.sort_values(by=["userId", "timestamp"])

user_seq = ratings.groupby("userId")["movieId"].apply(list)

item_set = ratings["movieId"].unique()
item2idx = {item: i+1 for i, item in enumerate(item_set)}
idx2item = {i: item for item, i in item2idx.items()}

user_sequences = []
for seq in user_seq:
    user_sequences.append([item2idx[i] for i in seq])

# 4. Dataset

In [17]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, max_len):
        self.sequences = sequences
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]

        seq = seq[-self.max_len:]
        input_seq = seq[:-1]
        target = seq[1:]

        pad_len = self.max_len - len(input_seq)
        input_seq = [0]*pad_len + input_seq
        target = [0]*pad_len + target

        return torch.LongTensor(input_seq), torch.LongTensor(target)

# 5. 모델

In [18]:
class SASRec(nn.Module):
    def __init__(self, num_items, hidden_dim=50, max_len=50, num_heads=2, num_layers=2):
        super(SASRec, self).__init__()

        self.item_emb = nn.Embedding(num_items+1, hidden_dim, padding_idx=0) #이게 학습되는 weight라고 생각하면된다 hidden_dim 이 50이면 50차원으로 나타낸거 10이라는 숫자를 [0.2 , 0.3 , ,,,, (총 50개) ] 이런식으로 이게 비슷한애들은 비슷해지도록 업데이트된다.
        self.pos_emb = nn.Embedding(max_len, hidden_dim)

        self.layers = nn.ModuleList([ #nn.ModuleList는 여러 개의 layer를 "리스트처럼 저장"하는 PyTorch 전용 구조
            nn.TransformerEncoderLayer( # Input -> Multi-Head Attention(Q,K,V 생성) -> Feed Forward -> Output
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim*4,
                dropout=0.2,
                batch_first=True
            )
            for _ in range(num_layers)
        ])

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, batch_input):
        batch_size, seq_len = batch_input.size()

        pos = torch.arange(seq_len, device=batch_input.device).unsqueeze(0).repeat(batch_size, 1)

        x_emb = self.item_emb(batch_input)
        x = x_emb + self.pos_emb(pos)
        x = self.dropout(x)

        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        padding_mask = (batch_input == 0)
        # padding_mask = (batch_input == 0)

        for layer in self.layers:
            x = layer(x, src_mask=mask, src_key_padding_mask=padding_mask)

        x = self.layer_norm(x)

        return x

# 6. 데이터 분리

In [19]:
train_sequences = []
val_data = []
test_data = []

for seq in user_sequences:
    if len(seq) < 15:
        continue

    train = seq[:-10]
    val   = seq[-10:-5]
    test  = seq[-5:]

    train_sequences.append(train)
    val_data.append((train + val[:-1], [val[-1]]))
    test_data.append((train + val, test))

# 7. 추천을 만들어내는 함수

In [20]:
def recommend(model, seq, top_k=10):
    model.eval()

    seq = seq[-max_len:]
    seq = [0]*(max_len-len(seq)) + seq
    seq = torch.LongTensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(seq)
        last = output[:, -1, :] #모든 Batch , 마지막시점, 모든 feature batch_first=True를 썼기때문에

        scores = torch.matmul(last, model.item_emb.weight.T)
        top_items = torch.topk(scores, top_k).indices.squeeze().cpu().numpy() # torch.topk 는 가장 큰 값 k 개 뽑기 , indices는 그것의 인덱스

    seen = set(seq)

    result = []
    for i in top_items:
        if i != 0 and idx2item[i] not in seen:
            result.append(idx2item[i])
        if len(result) == top_k:
            break
    return result

#8. 평가함수

In [21]:
def evaluate(model, data, top_k=10):
    HR, REC, NDCG, MAP = 0, 0, 0, 0

    for seq, gt in data:
        pred = recommend(model, seq, top_k)

        # HR
        HR += int(len(set(pred) & set(gt)) > 0)

        # Recall
        REC += len(set(pred) & set(gt)) / len(gt)

        # NDCG
        dcg = 0
        for i, p in enumerate(pred):
            if p in gt:
                dcg += 1 / torch.log2(torch.tensor(i+2.0))
        idcg = sum([1 / torch.log2(torch.tensor(i+2.0)) for i in range(min(len(gt), len(pred)))])
        NDCG += (dcg / idcg).item() if idcg > 0 else 0

        # MAP
        score = 0
        hit = 0
        for i, p in enumerate(pred):
            if p in gt:
                hit += 1
                score += hit / (i+1)
        MAP += score / len(gt) if len(gt) > 0 else 0

    n = len(data)
    return HR/n, REC/n, NDCG/n, MAP/n

# 9.학습


In [22]:
# “validation 성능이 더 이상 좋아지지 않으면 학습을 멈추기 위해”
best_val_ndcg = 0 #지금까지 나온 가장 좋은 성능 저장
patience = 3 #3번 연속 성능 안 좋아지면 멈춤
counter = 0 #성능이 안 좋아진 횟수 카운트


dataset = SASRecDataset(train_sequences, max_len)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SASRec(num_items=len(item2idx), max_len=max_len).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)


for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_input, batch_target in loader:
        batch_input = batch_input.to(device)
        batch_target = batch_target.to(device)

        output = model(batch_input) #이게 내부적으로 model.forward(batch_input) 자동호출

        output = output.reshape(-1, output.size(-1)) #output은 (128,50,50)이여서 reshape(-1,output.size(-1))하면
        # output.size가 (128,50,50) 이고 이거의 맨뒤는 50이니까 reshape(-1,50)이랑 같은거라 (6400,50)이 나옴
        batch_target = batch_target.reshape(-1) # batch_target은 (128,50) 이여서 reshape하면 (6400,)이 나옴

        # negative sampling
        neg_items = torch.randint(1, len(item2idx)+1, batch_target.shape).to(device) #torch.randint(low , high , size)  =  최소값 ,최대값 ,출력 tensor의 shape
        pos_emb = model.item_emb(batch_target)
        neg_emb = model.item_emb(neg_items)

        pos_logits = (output * pos_emb).sum(-1)
        neg_logits = (output * neg_emb).sum(-1)

        # padding 제외
        mask = batch_target != 0

        pos_logits = pos_logits[mask]
        neg_logits = neg_logits[mask]

        loss = -torch.log(torch.sigmoid(pos_logits - neg_logits)).mean()
#BPR loss는 정답 아이템이 가짜 아이템보다 점수가 높도록 학습하는 pairwise ranking loss이다.
#pos_logits - neg_logits를 통해 두 아이템의 점수 차이를 계산하고, 이를 sigmoid에 넣어 확률로 변환한다.
#이후 log를 취하고 음수를 붙여 loss로 만들면, 잘 맞춘 경우에는 loss가 작아지고 틀린 경우에는 loss가 크게 증가하도록 설계된다.

        optimizer.zero_grad() #기존 gradient 초기화 (0으로 reset)
        loss.backward() #미분계산
        optimizer.step() # 실제로 파라미터 업데이트

        total_loss += loss.item()

    val_hr, val_rec, val_ndcg, val_map = evaluate(model, val_data)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    print(f"Val HR@10: {val_hr:.4f}, Recall@10: {val_rec:.4f}, NDCG@10: {val_ndcg:.4f}, MAP@10: {val_map:.4f}")

    # Early stopping
    if val_ndcg > best_val_ndcg:
        best_val_ndcg = val_ndcg
        counter = 0
        best_model = model.state_dict()

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

model.load_state_dict(best_model)

Epoch 1, Loss: 159.5870
Val HR@10: 0.0015, Recall@10: 0.0015, NDCG@10: 0.0007, MAP@10: 0.0004
Epoch 2, Loss: 114.9290
Val HR@10: 0.0015, Recall@10: 0.0015, NDCG@10: 0.0007, MAP@10: 0.0005
Epoch 3, Loss: 85.6992
Val HR@10: 0.0017, Recall@10: 0.0017, NDCG@10: 0.0007, MAP@10: 0.0005
Epoch 4, Loss: 66.2822
Val HR@10: 0.0017, Recall@10: 0.0017, NDCG@10: 0.0007, MAP@10: 0.0005
Epoch 5, Loss: 52.1256
Val HR@10: 0.0013, Recall@10: 0.0013, NDCG@10: 0.0006, MAP@10: 0.0004
Epoch 6, Loss: 42.3619
Val HR@10: 0.0015, Recall@10: 0.0015, NDCG@10: 0.0007, MAP@10: 0.0004
Early stopping!


<All keys matched successfully>

# 10. 최종 테스트 평가

In [23]:
test_hr, test_rec, test_ndcg, test_map = evaluate(model, test_data)

print(f"Test HR@10: {test_hr:.4f}")
print(f"Test Recall@10: {test_rec:.4f}")
print(f"Test NDCG@10: {test_ndcg:.4f}")
print(f"Test MAP@10: {test_map:.4f}")


Test HR@10: 0.0071
Test Recall@10: 0.0014
Test NDCG@10: 0.0011
Test MAP@10: 0.0004
